# 城投转型 重构

## 1. 项目概述

### 1.1 功能描述

本项目用于城投融资平台转型数据管理，主要功能包括：
- 跟踪城投融资平台转型情况
- 记录市场化退出过程
- 数据汇总与统计分析

### 1.2 数据源

- 数据库：MySQL (yq数据库)
- 表名：转型城投清单 / 城投平台退出

### 1.3 输出结果

- 转型城投清单 JSON/CSV 文件
- 各维度统计分析报告

---

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import json
import datetime
from typing import Dict, List, Optional, Any

# 第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text

# 配置模块
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import Config

print("依赖导入完成")
print(f"pandas版本: {pd.__version__}")
print(f"sqlalchemy版本: {sqlalchemy.__version__}")

### 2.2 配置参数

In [ ]:
# 加载配置
config = Config()

# 数据库配置
DB_HOST = config.db_host
DB_PORT = config.db_port
DB_NAME = config.db_name
DB_USER = config.db_user
DB_PASSWORD = config.db_password

# 输出配置
OUTPUT_DIR = config.output_dir
DATA_DIR = config.data_dir

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"数据库主机: {DB_HOST}")
print(f"数据库名称: {DB_NAME}")
print(f"输出目录: {OUTPUT_DIR}")

---

## 3. 数据获取

### 3.1 数据库连接

In [ ]:
def create_db_engine():
    """创建数据库连接引擎"""
    connection_string = f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4'
    engine = sqlalchemy.create_engine(
        connection_string,
        pool_recycle=3600,
        pool_pre_ping=True,
        echo=False
    )
    return engine

# 创建引擎
engine = create_db_engine()
print("数据库连接引擎创建成功")

In [ ]:
def test_connection():
    """测试数据库连接"""
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT 1"))
            print("数据库连接测试成功")
            return True
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return False

test_connection()

### 3.2 转型城投数据查询

In [ ]:
def query_transformation_data(engine, table_name='城投平台市场化转型') -> pd.DataFrame:
    """
    查询城投转型数据
    
    Parameters:
    -----------
    engine: SQLAlchemy引擎
    table_name: 表名
    
    Returns:
    --------
    DataFrame: 转型城投数据
    """
    query = f"""
    SELECT 
        itName as 公司名称,
        itCode2 as 公司代码,
        regionName as 区域名称,
        regionCode as 区域代码,
        releaseDate as 发布日期,
        changeType as 变更类型,
        changeReason as 变更原因,
        reasonDescription as 原因描述,
        publishEntity as 发布主体,
        url as 文件链接,
        guid as 唯一标识,
        sourceFlag as 来源标志,
        subjectRating as 主体评级,
        bondStockSize as 债券库存规模,
        issueBond as 是否发债,
        actualController as 实际控制人,
        actualControllerCode as 实际控制人代码,
        province as 省份,
        city as 城市,
        county as 区县,
        description as 描述,
        trSimpleName as 交易简称,
        trCode as 交易代码,
        symbol as 股票代码
    FROM {table_name}
    ORDER BY releaseDate DESC
    """
    
    try:
        df = pd.read_sql(query, engine)
        print(f"成功查询 {len(df)} 条转型城投记录")
        return df
    except Exception as e:
        print(f"查询失败: {e}")
        return pd.DataFrame()

# 执行查询
df_raw = query_transformation_data(engine)
df_raw.head()

---

## 4. 数据处理

### 4.1 数据验证

In [ ]:
def validate_data(df: pd.DataFrame) -> Dict[str, Any]:
    """
    验证数据完整性
    
    Parameters:
    -----------
    df: 待验证的DataFrame
    
    Returns:
    --------
    Dict: 验证结果
    """
    validation_result = {
        'total_records': len(df),
        'missing_required': {},
        'invalid_dates': [],
        'data_issues': []
    }
    
    # 必填字段检查
    required_fields = ['公司名称', '公司代码', '区域名称', '区域代码', '发布日期', '变更类型', '变更原因', '发布主体', '唯一标识', '省份', '城市']
    
    for field in required_fields:
        if field in df.columns:
            missing_count = df[field].isna().sum()
            if missing_count > 0:
                validation_result['missing_required'][field] = missing_count
    
    # 日期格式检查
    if '发布日期' in df.columns:
        for idx, date_val in df['发布日期'].items():
            if pd.notna(date_val):
                try:
                    if isinstance(date_val, str):
                        datetime.datetime.strptime(date_val, '%Y-%m-%d')
                except ValueError:
                    validation_result['invalid_dates'].append(idx)
    
    # 条件字段检查：发债公司应有评级
    if '是否发债' in df.columns and '主体评级' in df.columns:
        bond_without_rating = df[(df['是否发债'] == '是') & (df['主体评级'].isna())]
        if len(bond_without_rating) > 0:
            validation_result['data_issues'].append(f"{len(bond_without_rating)}家发债公司缺失评级信息")
    
    return validation_result

# 执行验证
validation = validate_data(df_raw)
print("数据验证结果:")
for key, value in validation.items():
    print(f"  {key}: {value}")

### 4.2 数据清洗

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    清洗数据
    
    Parameters:
    -----------
    df: 原始DataFrame
    
    Returns:
    --------
    DataFrame: 清洗后的数据
    """
    df_clean = df.copy()
    
    # 标准化字符串字段
    string_columns = df_clean.select_dtypes(include=['object']).columns
    for col in string_columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].replace('null', np.nan)
        df_clean[col] = df_clean[col].replace('None', np.nan)
    
    # 转换日期字段
    if '发布日期' in df_clean.columns:
        df_clean['发布日期'] = pd.to_datetime(df_clean['发布日期'], errors='coerce')
    
    # 转换数值字段
    if '债券库存规模' in df_clean.columns:
        df_clean['债券库存规模'] = pd.to_numeric(df_clean['债券库存规模'], errors='coerce')
    
    # 添加年月字段便于统计分析
    if '发布日期' in df_clean.columns:
        df_clean['发布年月'] = df_clean['发布日期'].dt.strftime('%Y-%m')
        df_clean['发布年份'] = df_clean['发布日期'].dt.year
        df_clean['发布月份'] = df_clean['发布日期'].dt.month
    
    print(f"数据清洗完成，共 {len(df_clean)} 条记录")
    return df_clean

# 执行清洗
df = clean_data(df_raw)
df.info()

---

## 5. 核心逻辑

### 5.1 数据统计与汇总

In [ ]:
def generate_summary(df: pd.DataFrame) -> Dict[str, Any]:
    """
    生成数据汇总统计
    
    Parameters:
    -----------
    df: 清洗后的DataFrame
    
    Returns:
    --------
    Dict: 统计结果
    """
    summary = {
        'total': len(df),
        'companyNum': df['公司名称'].nunique() if '公司名称' in df.columns else 0,
        'date_range': None,
        'bond_total_size': 0
    }
    
    # 日期范围
    if '发布日期' in df.columns:
        min_date = df['发布日期'].min()
        max_date = df['发布日期'].max()
        if pd.notna(min_date) and pd.notna(max_date):
            summary['date_range'] = f"{min_date.strftime('%Y-%m-%d')} ~ {max_date.strftime('%Y-%m-%d')}"
    
    # 债券总规模
    if '债券库存规模' in df.columns:
        summary['bond_total_size'] = df['债券库存规模'].sum()
    
    return summary

# 生成汇总
summary = generate_summary(df)
print("数据汇总:")
for key, value in summary.items():
    print(f"  {key}: {value}")

### 5.2 变更类型分析

In [ ]:
def analyze_change_type(df: pd.DataFrame) -> pd.DataFrame:
    """
    按变更类型统计
    
    变更类型说明:
    - 退出: 城投融资平台完全退出政府融资职能
    - 市场化转型: 按市场化要求建立公司法人治理结构
    - 隐性债务清零: 隐性债务已清零，退出融资平台监管范畴
    """
    if '变更类型' not in df.columns:
        return pd.DataFrame()
    
    stats = df.groupby('变更类型').agg({
        '公司名称': 'count',
        '债券库存规模': 'sum'
    }).rename(columns={'公司名称': '数量', '债券库存规模': '债券规模合计'})
    
    stats['占比'] = (stats['数量'] / stats['数量'].sum() * 100).round(2)
    stats = stats.sort_values('数量', ascending=False)
    
    return stats

# 变更类型统计
change_type_stats = analyze_change_type(df)
print("\n变更类型统计:")
print(change_type_stats)

In [ ]:
def analyze_change_reason(df: pd.DataFrame) -> pd.DataFrame:
    """
    按变更原因统计
    
    变更原因说明:
    - 市场化转型: 转型为市场化运作的国有企业
    - 隐性债务清零: 实现隐性债务和融资平台双清零
    """
    if '变更原因' not in df.columns:
        return pd.DataFrame()
    
    stats = df.groupby('变更原因').agg({
        '公司名称': 'count'
    }).rename(columns={'公司名称': '数量'})
    
    stats['占比'] = (stats['数量'] / stats['数量'].sum() * 100).round(2)
    stats = stats.sort_values('数量', ascending=False)
    
    return stats

# 变更原因统计
change_reason_stats = analyze_change_reason(df)
print("\n变更原因统计:")
print(change_reason_stats)

### 5.3 区域分析

In [ ]:
def analyze_by_province(df: pd.DataFrame) -> pd.DataFrame:
    """
    按省份统计
    """
    if '省份' not in df.columns:
        return pd.DataFrame()
    
    stats = df.groupby('省份').agg({
        '公司名称': 'count',
        '债券库存规模': 'sum'
    }).rename(columns={'公司名称': '数量', '债券库存规模': '债券规模合计'})
    
    stats['占比'] = (stats['数量'] / stats['数量'].sum() * 100).round(2)
    stats = stats.sort_values('数量', ascending=False)
    
    return stats

def analyze_by_city(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """
    按城市统计TOP N
    """
    if '城市' not in df.columns:
        return pd.DataFrame()
    
    stats = df.groupby(['省份', '城市']).agg({
        '公司名称': 'count',
        '债券库存规模': 'sum'
    }).rename(columns={'公司名称': '数量', '债券库存规模': '债券规模合计'})
    
    stats = stats.sort_values('数量', ascending=False).head(top_n)
    
    return stats

# 省份统计
province_stats = analyze_by_province(df)
print("\n省份统计TOP10:")
print(province_stats.head(10))

# 城市统计
city_stats = analyze_by_city(df)
print("\n城市统计TOP20:")
print(city_stats)

In [ ]:
def analyze_publish_entity(df: pd.DataFrame) -> pd.DataFrame:
    """
    按发布主体统计
    
    发布主体说明:
    - 企业披露: 企业主动披露转型信息 (sourceFlag=2)
    - 官方披露: 政府或监管部门披露 (sourceFlag=1)
    - 其他: 其他方式披露
    """
    if '发布主体' not in df.columns:
        return pd.DataFrame()
    
    stats = df.groupby('发布主体').agg({
        '公司名称': 'count'
    }).rename(columns={'公司名称': '数量'})
    
    stats['占比'] = (stats['数量'] / stats['数量'].sum() * 100).round(2)
    stats = stats.sort_values('数量', ascending=False)
    
    return stats

# 发布主体统计
publish_entity_stats = analyze_publish_entity(df)
print("\n发布主体统计:")
print(publish_entity_stats)

In [ ]:
def analyze_rating(df: pd.DataFrame) -> pd.DataFrame:
    """
    按主体评级统计
    
    评级等级:
    - AAA: 最高级
    - AA+: 高级
    - AA: 中高级
    - AA-: 中级
    - A: 中低级
    """
    if '主体评级' not in df.columns:
        return pd.DataFrame()
    
    # 过滤有效评级
    df_with_rating = df[df['主体评级'].notna()]
    
    stats = df_with_rating.groupby('主体评级').agg({
        '公司名称': 'count',
        '债券库存规模': 'sum'
    }).rename(columns={'公司名称': '数量', '债券库存规模': '债券规模合计'})
    
    stats['占比'] = (stats['数量'] / stats['数量'].sum() * 100).round(2)
    
    # 按评级等级排序
    rating_order = ['AAA', 'AA+', 'AA', 'AA-', 'A']
    stats = stats.reindex([r for r in rating_order if r in stats.index])
    
    return stats

# 评级统计
rating_stats = analyze_rating(df)
print("\n主体评级统计:")
print(rating_stats)

In [ ]:
def analyze_monthly_trend(df: pd.DataFrame) -> pd.DataFrame:
    """
    按月度统计趋势
    """
    if '发布年月' not in df.columns:
        return pd.DataFrame()
    
    stats = df.groupby('发布年月').agg({
        '公司名称': 'count',
        '债券库存规模': 'sum'
    }).rename(columns={'公司名称': '数量', '债券库存规模': '债券规模合计'})
    
    stats = stats.sort_index()
    
    return stats

# 月度趋势
monthly_stats = analyze_monthly_trend(df)
print("\n月度趋势:")
print(monthly_stats)

---

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def main():
    """
    主执行函数
    """
    print("="*60)
    print("城投转型数据分析")
    print("="*60)
    
    # 1. 数据查询
    print("\n[1] 数据查询...")
    df_raw = query_transformation_data(engine)
    
    if df_raw.empty:
        print("未获取到数据，退出执行")
        return
    
    # 2. 数据验证
    print("\n[2] 数据验证...")
    validation = validate_data(df_raw)
    print(f"验证结果: {validation}")
    
    # 3. 数据清洗
    print("\n[3] 数据清洗...")
    df = clean_data(df_raw)
    
    # 4. 数据统计
    print("\n[4] 数据统计...")
    summary = generate_summary(df)
    print(f"汇总统计: {summary}")
    
    # 5. 各维度分析
    print("\n[5] 多维度分析...")
    stats_results = {
        '变更类型': analyze_change_type(df),
        '变更原因': analyze_change_reason(df),
        '省份分布': analyze_by_province(df),
        '城市TOP20': analyze_by_city(df),
        '发布主体': analyze_publish_entity(df),
        '主体评级': analyze_rating(df),
        '月度趋势': analyze_monthly_trend(df)
    }
    
    for name, stats_df in stats_results.items():
        if not stats_df.empty:
            print(f"\n{name}:")
            print(stats_df)
    
    print("\n" + "="*60)
    print("执行完成")
    print("="*60)
    
    return df, stats_results

# 执行主流程
df_result, stats_result = main()

### 6.2 数据导出

In [ ]:
def export_to_json(df: pd.DataFrame, output_path: str) -> None:
    """
    导出为JSON格式
    """
    # 构建JSON结构
    export_data = {
        'total': len(df),
        'companyNum': df['公司名称'].nunique() if '公司名称' in df.columns else 0,
        'exportTime': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'data': df.to_dict(orient='records')
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2, default=str)
    
    print(f"JSON文件已导出: {output_path}")

def export_to_csv(df: pd.DataFrame, output_path: str) -> None:
    """
    导出为CSV格式
    """
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"CSV文件已导出: {output_path}")

def export_stats_to_csv(stats_dict: Dict[str, pd.DataFrame], output_dir: str) -> None:
    """
    导出统计结果到CSV
    """
    for name, stats_df in stats_dict.items():
        if not stats_df.empty:
            output_path = os.path.join(output_dir, f'{name}.csv')
            stats_df.to_csv(output_path, encoding='utf-8-sig')
            print(f"统计文件已导出: {output_path}")

# 导出数据
if not df_result.empty:
    export_to_json(df_result, os.path.join(OUTPUT_DIR, '转型城投清单.json'))
    export_to_csv(df_result, os.path.join(OUTPUT_DIR, '转型城投清单.csv'))
    export_stats_to_csv(stats_result, OUTPUT_DIR)

---

## 7. 工具函数（可复用）

In [ ]:
def get_company_detail(df: pd.DataFrame, company_name: str) -> Optional[pd.Series]:
    """
    获取指定公司的详细信息
    
    Parameters:
    -----------
    df: 数据DataFrame
    company_name: 公司名称
    
    Returns:
    --------
    Series or None: 公司详细信息
    """
    if '公司名称' not in df.columns:
        return None
    
    result = df[df['公司名称'] == company_name]
    if len(result) > 0:
        return result.iloc[0]
    return None

def filter_by_region(df: pd.DataFrame, province: str = None, city: str = None) -> pd.DataFrame:
    """
    按区域筛选数据
    
    Parameters:
    -----------
    df: 数据DataFrame
    province: 省份名称
    city: 城市名称
    
    Returns:
    --------
    DataFrame: 筛选结果
    """
    result = df.copy()
    
    if province and '省份' in result.columns:
        result = result[result['省份'] == province]
    
    if city and '城市' in result.columns:
        result = result[result['城市'] == city]
    
    return result

def filter_by_date_range(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    """
    按日期范围筛选数据
    
    Parameters:
    -----------
    df: 数据DataFrame
    start_date: 开始日期 (YYYY-MM-DD)
    end_date: 结束日期 (YYYY-MM-DD)
    
    Returns:
    --------
    DataFrame: 筛选结果
    """
    if '发布日期' not in df.columns:
        return df
    
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    return df[(df['发布日期'] >= start) & (df['发布日期'] <= end)]

def filter_by_rating(df: pd.DataFrame, ratings: List[str]) -> pd.DataFrame:
    """
    按评级筛选数据
    
    Parameters:
    -----------
    df: 数据DataFrame
    ratings: 评级列表，如 ['AAA', 'AA+']
    
    Returns:
    --------
    DataFrame: 筛选结果
    """
    if '主体评级' not in df.columns:
        return df
    
    return df[df['主体评级'].isin(ratings)]

print("工具函数加载完成")